In [17]:
import pandas as pd
from pandasql import sqldf

In [18]:
users = pd.DataFrame({
    'user_id': [1,2,3,4],
    'name': ['Аня','Боря','Вера','Гена']
})
orders = pd.DataFrame({
    'order_id': [101, 102, 103],
    'user_id': [1, 1, 2],
    'amount': [500, 300, 700]
})

sqldf("SELECT users.name, orders.amount FROM users INNER JOIN orders ON users.user_id = orders.user_id")

,name,amount
0,Аня,300
1,Аня,500
2,Боря,700


In [19]:
sqldf("SELECT users.name, orders.amount FROM users LEFT JOIN orders ON users.user_id = orders.user_id")

,name,amount
0,Аня,300.0
1,Аня,500.0
2,Боря,700.0
3,Вера,NaN
4,Гена,NaN


In [20]:
sqldf("SELECT users.name FROM users LEFT JOIN orders ON users.user_id = orders.user_id WHERE orders.order_id is NULL")

,name
0,Вера
1,Гена


In [21]:
reviews = pd.read_csv('/home/art/olist_data/olist_order_reviews_dataset.csv')
reviews.head()
order_items = pd.read_csv('/home/art/olist_data/olist_order_items_dataset.csv')

In [22]:
reviews.shape

(99224, 7)

In [23]:
orders = pd.read_csv('/home/art/olist_data/olist_orders_dataset.csv')

sqldf("""
    SELECT orders.order_id, orders.order_status, reviews.review_score
    FROM orders
    LEFT JOIN reviews ON orders.order_id = reviews.order_id
    LIMIT 10
""")

,order_id,order_status,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,4
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,4
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,5
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,5
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,5
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,4
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,2
7,6514b8ad8028c9f2cc2374ded245783f,delivered,5
8,76c6e866289321a7c93b82b54852dc33,delivered,1
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,5


In [24]:
sqldf("""
    SELECT COUNT(*) AS orders_without_reviews
    FROM orders
    LEFT JOIN reviews ON orders.order_id = reviews.order_id
    WHERE reviews.review_id IS NULL
""")

,orders_without_reviews
0,768


In [25]:
sqldf("""
    SELECT COUNT(*) AS orders_with_reviews
    FROM orders
    INNER JOIN reviews ON orders.order_id = reviews.order_id
""")

,orders_with_reviews
0,99224


In [26]:
sqldf("SELECT AVG(review_score) AS avg_score FROM reviews")

,avg_score
0,4.086421


In [28]:

customers = pd.read_csv('/home/art/olist_data/olist_customers_dataset.csv')

sqldf("""
    SELECT 
        customers.customer_state,
        SUM(order_items.price) AS revenue
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    JOIN order_items ON orders.order_id = order_items.order_id
    GROUP BY customers.customer_state
    ORDER BY revenue DESC
    LIMIT 10
""")

,customer_state,revenue
0,SP,5202955.05
1,RJ,1824092.67
2,MG,1585308.03
3,RS,750304.02
4,PR,683083.76
5,SC,520553.34
6,BA,511349.99
7,DF,302603.94
8,GO,294591.95
9,ES,275037.31


In [29]:
sqldf("""
    SELECT
        customers.customer_city,
        COUNT(DISTINCT orders.order_id) AS orders_count
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    GROUP BY customers.customer_city
    ORDER BY orders_count DESC
    LIMIT 10
""")

,customer_city,orders_count
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
5,campinas,1444
6,porto alegre,1379
7,salvador,1245
8,guarulhos,1189
9,sao bernardo do campo,938


In [32]:
sqldf("""
    SELECT
        customers.customer_state,
        AVG(order_items.price) AS avg_check
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    JOIN order_items ON orders.order_id = order_items.order_id
    GROUP BY customers.customer_state
    ORDER BY avg_check DESC
    LIMIT 10
""")

,customer_state,avg_check
0,PB,191.475216
1,AL,180.889212
2,AC,173.727717
3,RO,165.973525
4,PA,165.692417
5,AP,164.320732
6,PI,160.358081
7,TO,157.529333
8,RN,156.965936
9,CE,153.758261


In [41]:
sqldf("""
    SELECT
        customers.customer_state,
        COUNT(DISTINCT customers.customer_unique_id) AS unique_customers
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    GROUP BY customers.customer_state
    ORDER BY unique_customers DESC
    LIMIT 10
""")

,customer_state,unique_customers
0,SP,40302
1,RJ,12384
2,MG,11259
3,RS,5277
4,PR,4882
5,SC,3534
6,BA,3277
7,DF,2075
8,ES,1964
9,GO,1952


In [45]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [47]:
sqldf("""
    SELECT
        COUNT(*) AS five_stars
    FROM reviews
    WHERE review_score = '5'
""")

,five_stars
0,57328


In [49]:
sqldf("""
    SELECT
        COUNT(DISTINCT customers.customer_city) AS unique_city
    FROM customers
""")

,unique_city
0,4119


In [51]:
sqldf("""
    SELECT AVG(order_items.price) AS avg_check
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    JOIN order_items ON orders.order_id = order_items.order_id
    WHERE customers.customer_state = 'SP'
""")

,avg_check
0,109.653629


In [52]:
sqldf("""
    SELECT
        order_items.seller_id,
        COUNT(*) AS count
    FROM order_items
    GROUP BY seller_id
    ORDER BY count DESC
    LIMIT 5
""")

,seller_id,count
0,6560211a19b47992c3666cc44a7e94c0,2033
1,4a3ca9315b744ce9f8e9374361493884,1987
2,1f50f920176fa81dab994f9023523100,1931
3,cc419e0650a3c5ba77189a1882b7556a,1775
4,da8622b14eb17ae2831f4ac5b9dab84a,1551


In [56]:
sqldf("""
    SELECT
        COUNT(DISTINCT orders.order_id) AS count
    FROM orders
    WHERE order_status = 'canceled' AND order_purchase_timestamp >= '2018-01-01' AND order_purchase_timestamp < '2019-01-01'
""")

,count
0,334


In [57]:
sqldf("""
    SELECT order_id, price
    FROM order_items
    WHERE price > (SELECT AVG(price) AS avg_price FROM order_items)
    LIMIT 10
""")

,order_id,price
0,00018f77f2f0320c557190d7a144bdd3,239.90
1,000229ec398224ef6ca0657da4fc703e,199.00
2,00042b26cf59d7ce69dfabb4e55b4fd9,199.90
3,000576fe39319847cbb9d288c5617fa6,810.00
4,0005a1a1728c9d785b8e2b08b904576c,145.95
5,0009c9a17f916a706d71784483a5d643,639.00
6,000aed2e25dbad2f9ddb70584c5a2ded,144.00
7,00119ff934e539cf26f92b9ef0cdfed8,219.90
8,0011d82c4b53e22e84023405fb467e57,289.00
9,00137e170939bba5a3134e2386413108,397.00


In [59]:
sqldf("""
    SELECT AVG(order_total) AS avg_order_total
    FROM(
        SELECT order_id, SUM(price) AS order_total
        FROM order_items
        GROUP BY order_id
    )
""")

,avg_order_total
0,137.754076


In [60]:
sqldf("""
    SELECT
        order_id,
        price,
        price - (SELECT AVG(price) FROM order_items) AS diff_from_avg
    FROM order_items
    LIMIT 10
""")

,order_id,price,diff_from_avg
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,-61.753739
1,00018f77f2f0320c557190d7a144bdd3,239.90,119.246261
2,000229ec398224ef6ca0657da4fc703e,199.00,78.346261
3,00024acbcdf0a6daa1e931b038114c75,12.99,-107.663739
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,79.246261
5,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,-98.753739
6,00054e8431b9d7675808bcb819fb4a32,19.90,-100.753739
7,000576fe39319847cbb9d288c5617fa6,810.00,689.346261
8,0005a1a1728c9d785b8e2b08b904576c,145.95,25.296261
9,0005f50442cb953dcd1d21e1fb923495,53.99,-66.663739


In [62]:
sqldf("""
    WITH order_totals AS (
        SELECT order_id, SUM(price) AS order_total
        FROM order_items
        GROUP BY order_id
    )
    SELECT AVG(order_total) AS avg_order_total
    FROM order_totals
""")

,avg_order_total
0,137.754076


In [66]:
sqldf("""
    WITH customer_revenue AS (
        SELECT
            orders.customer_id,
            SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY orders.customer_id
    )
    SELECT customer_id, revenue
    FROM customer_revenue
    ORDER BY revenue DESC
    LIMIT 10
""")

,customer_id,revenue
0,1617b1357756262bfa56ab541c47bc16,13440.0
1,ec5b2ba62e574342386871631fafd3fc,7160.0
2,c6e2731c5b391845f6800c97401a43a9,6735.0
3,f48d464a0baaea338cb25f816991ab1f,6729.0
4,3fd6777bbce08a352fddd04e4a7cc8f6,6499.0
5,05455dfa7cd02f13d132aa7a6a9729c6,5934.6
6,df55c14d1476a9a3467f131269c2477f,4799.0
7,24bbf5fd2f2e1b359ee7de94defc4a15,4690.0
8,e0a2412720e9ea4f26c1ac985f6a7358,4599.9
9,3d979689f636322c62418b6346b1c6d2,4590.0
